In [229]:
import sys
from pathlib import Path
sys.path.append(str(Path('../python_code').resolve()))
import numpy as np
import cv2
import csv
from scipy.stats import rankdata
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
%matplotlib inline
from DataGenerator import HyperspectralTorchDataset
from BandAttentionModel import BandAttentionModel

In [230]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image_path_infected = Path("../home/ARO.local/collaboration/sagi-tomer-collab/Normalized_Tomato_Seeds/Infected/Tray_2_row_0_column_12.npy")
image_path_healthy = Path("../home/ARO.local/collaboration/sagi-tomer-collab/Normalized_Tomato_Seeds/Healthy/Tray_2_row_17_column_8.npy")
model_path = Path("../home/ARO.local/collaboration/sagi-tomer-collab/Normalized_Tomato_Seeds/Models/BandAttentionModel_1-64-127-190-253-316-379-442-505-568-631-bands-0.825-accuracy.pt")
# model_path = Path("../home/ARO.local/collaboration/sagi-tomer-collab/Normalized_Tomato_Seeds/Models/BandAttentionModel_1-64-127-190-253-316-379-442-505-568-631-bands-0.957-accuracy.pt")
seeds_root = Path("../home/ARO.local/collaboration/sagi-tomer-collab/Normalized_Tomato_Seeds")

# --- Infer Bands and Label ---
model_name = model_path.stem
bands_str = model_name.split("_")[1].split("-bands")[0]
bands = list(map(int, bands_str.split("-")))
# label = 0 if "Healthy" in str(image_path) else 1
# print(bands, label)

# --- Prepare Image ---
image_infected = np.load(image_path_infected)
height, width = image_infected.shape[:2]
shape = (height, width, len(bands))

# dataset = HyperspectralTorchDataset([str(image_path)], [label], bands, shape)
dataset = HyperspectralTorchDataset([str(image_path_infected), str(image_path_healthy)], [1,0], bands, shape)
loader = DataLoader(dataset, batch_size=2, shuffle=False)

for image, _ in loader:
    image = image.to(device)

In [231]:
attention_model = BandAttentionModel(len(bands))

# Load the state dict
state = torch.load(model_path, map_location=device)

# Strip the prefix, if needed (for example, '_orig_mod.')
state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}

# Get current model state_dict
model_state = attention_model.state_dict()

# Update only the matching keys
for key in model_state.keys():
    if key in state and state[key].shape == model_state[key].shape:
        model_state[key] = state[key]  # Copy the matching weights

# Load the state_dict into the model
attention_model.load_state_dict(model_state)

# Now the model is partially loaded with compatible weights
# model = torch.compile(model, backend="eager")
attention_model.to(device).eval()

logits, attn_weights = attention_model(image)

C:\Users\sagig\AppData\Local\Temp\ipykernel_42072\944147639.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(model_path, map_location=device)


In [232]:
print(attn_weights.shape)
avg_attn = attn_weights.mean(dim=0)
print(avg_attn.shape)

torch.Size([2, 11, 11])
torch.Size([11, 11])


In [233]:
import itertools
import numpy as np

def best_and_worst_n_band_combinations(avg_attn: np.ndarray, n: int = 3):
    C = avg_attn.shape[0]
    best_score = -np.inf
    best_combo = None
    worst_score = np.inf
    worst_combo = None

    for combo in itertools.combinations(range(C), n):
        submatrix = avg_attn[np.ix_(combo, combo)]
        score = submatrix.sum()

        if score > best_score:
            best_score = score
            best_combo = combo

        if score < worst_score:
            worst_score = score
            worst_combo = combo
    print(submatrix)
    return best_combo, best_score, worst_combo, worst_score

# Usage example:
best_bands, best_score, worst_bands, worst_score = best_and_worst_n_band_combinations(avg_attn, n=4)
print("Best 3 bands:", best_bands, "with score:", best_score)
print("Worst 3 bands:", worst_bands, "with score:", worst_score)


tensor([[0.0948, 0.1037, 0.1151, 0.1224],
        [0.0950, 0.1037, 0.1147, 0.1217],
        [0.0954, 0.1038, 0.1144, 0.1210],
        [0.0956, 0.1038, 0.1142, 0.1207]], grad_fn=<IndexBackward0>)
Best 3 bands: (4, 8, 9, 10) with score: tensor(1.9339, grad_fn=<SumBackward0>)
Worst 3 bands: (1, 2, 3, 5) with score: tensor(0.8199, grad_fn=<SumBackward0>)
